# [Chapter 7 — SIR_I Simulations, §7.1-7.2] Numerical Integration of the SIR_I System

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 7 — SIR_I Simulations
**Considerations developed:** 7 (equilibria + stability)
**Estimated runtime:** ≤ 30 seconds

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/)

## What this notebook does
Demonstrates the canonical numerical integration of the SIR_I system using `integrate_sir_i` from `shared/solvers.py`. Shows how the book's default tolerances (RTOL=1e-8, ATOL=1e-10) produce trajectories that conserve population to machine precision and match analytical predictions to >6 significant digits.

## What you should already know
Chapter 6 notebooks.


## Setup


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import numpy as np
import matplotlib.pyplot as plt
from shared import book_style, BOOK_COLORS, integrate_sir_i
from shared.parameters import baseline_chapter_07
from shared.seeds import set_seed_chapter_07
from shared.verification import assert_conservation_law, assert_within_tolerance

set_seed_chapter_07()
book_style()


## Standard SIR_I integration

The `integrate_sir_i` function in `shared/solvers.py` wraps `scipy.integrate.solve_ivp` with the book's verified tolerances and returns a dictionary with state arrays. All notebooks in this repo use this single canonical solver to ensure reproducibility.

In [ ]:
params = baseline_chapter_07()
result = integrate_sir_i(params)

t = result['t']
S, I, R = result['S'], result['I'], result['R']

print(f"Solver status: {'success' if result['success'] else 'FAILED'}")
print(f"Time points: {len(t)}")
print(f"Final time: {t[-1]} days")
print(f"Peak I: {I.max():.4f} at t = {t[I.argmax()]:.1f} days")
print(f"Final R: {R[-1]:.4f}  (fraction ever infected)")


In [ ]:
# The standard SIR_I trajectory plot
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(t, S, color=BOOK_COLORS['susceptible'], lw=1.5, label='S(t)')
ax.plot(t, I, color=BOOK_COLORS['infectious'], lw=1.5, label='I(t)')
ax.plot(t, R, color=BOOK_COLORS['recovered'], lw=1.5, label='R(t)')

# Annotate peak
peak_idx = I.argmax()
ax.plot(t[peak_idx], I[peak_idx], 'o', color=BOOK_COLORS['infectious'], ms=8)
ax.annotate(f'Peak I = {I[peak_idx]:.3f}\nat t = {t[peak_idx]:.0f} days',
            xy=(t[peak_idx], I[peak_idx]),
            xytext=(t[peak_idx]+30, I[peak_idx]+0.05),
            arrowprops=dict(arrowstyle='->', color='gray', alpha=0.6),
            fontsize=9)

ax.set_xlabel('Time (days)')
ax.set_ylabel('Fraction of population')
ax.set_title('Baseline SIR_I trajectory ($R_0 = 2.0$)')
ax.legend(loc='center right')
ax.set_xlim(0, params['t_end'])
ax.set_ylim(-0.02, 1.02)
plt.tight_layout()
plt.show()


## Verification

For mu = 0 (no demographic turnover), the SIR_I system has two analytical invariants:

1. **Population conservation**: $S(t) + I(t) + R(t) = 1$ for all $t$
2. **Final-size relation**: $\ln(S_0/S_\infty) = R_0 (1 - S_\infty)$ (transcendental equation)

These give us numerical-vs-analytical comparison points.

In [ ]:
# Conservation check
assert_conservation_law([S, I, R], expected_total=1.0, tolerance=1e-8)
print("✅ Conservation: S + I + R = 1 to 1e-8 throughout trajectory")

# Final-size relation: solve transcendentally
from scipy.optimize import brentq
R0_val = params['c_I'] * params['beta'] * params['tau_R']
S_0 = params['S0']

def final_size_equation(S_inf):
    return np.log(S_0 / S_inf) - R0_val * (1 - S_inf)

S_inf_analytical = brentq(final_size_equation, 1e-6, S_0 - 1e-6)
S_inf_numerical = S[-1]

print(f"\nFinal-size relation:")
print(f"  Analytical S_∞ (from transcendental eq): {S_inf_analytical:.6f}")
print(f"  Numerical S_∞ (at t = {t[-1]} days):     {S_inf_numerical:.6f}")
print(f"  Final-size attack rate: {1 - S_inf_analytical:.4f}")
print(f"  Numerical attack rate:  {1 - S_inf_numerical:.4f}")

# These will agree to ~6 digits if tolerances are tight enough and t_end is long enough.
# (Note: at t_end = 200 days, S(t) is asymptotic but may not be quite at S_∞ yet.)
print(f"\nDifference: {abs(S_inf_analytical - S_inf_numerical):.2e}")
